In [1]:
# %pip install python-dotenv
# %uv add dspy

In [2]:
import os
# os.environ['OPENAI_API_KEY'] = input()
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

True

In [3]:
import dspy
lm = dspy.LM("azure/gpt-4.1", temperature=1.0, num_retries=0)
tlm = dspy.LM("azure/gpt-4.1",temperature=1.0)
dspy.configure(lm=lm)

In [4]:
print(lm("Say this is a test!") ) # => ['This is a test!']
print(lm(messages=[{"role": "user", "content": "Say this is a test!"}]))  # => ['This is a test!']
print(tlm(messages=[{"role": "user", "content": "Say this is a test!"}]))  # => ['This is a test!']

['This is a test!']
['This is a test!']
['This is a test!']


## Load the benchmark and view one example from the benchmark

In [5]:
from gepa_artifact.benchmarks.livebench_math import benchmark as lb_math_metas

In [6]:
bench = lb_math_metas[0].benchmark()

In [7]:
len(bench.train_set), len(bench.val_set), len(bench.test_set)

(121, 121, 126)

In [8]:
import pprint
pprint.pprint(bench.train_set[0])

Example({'question': 'What is the degree measure of the acute angle formed by lines with slopes $2$ and $\\tfrac{1}{3}$? $\\textbf{(A)}~45\\qquad\\textbf{(B)}~30\\qquad\\textbf{(C)}~52.5\\qquad\\textbf{(D)}~60\\qquad\\textbf{(E)}~37.5$ If you cannot determine the correct multiple-choice answer, take your best guess. Once you have your answer, please duplicate that letter five times in a single string. For example, if the answer is F, then write FFFFF.', 'answer': 'A', 'question_d': {'question_id': '07a0d7af7b149f35d0441e3b732fc4706ac286fca96748d7c3e4ceb95af46558', 'category': 'math', 'task': 'math_comp', 'subtask': 'updated_amc_12a_2023', 'year': '', 'turns': ['What is the degree measure of the acute angle formed by lines with slopes $2$ and $\\tfrac{1}{3}$? $\\textbf{(A)}~45\\qquad\\textbf{(B)}~30\\qquad\\textbf{(C)}~52.5\\qquad\\textbf{(D)}~60\\qquad\\textbf{(E)}~37.5$ If you cannot determine the correct multiple-choice answer, take your best guess. Once you have your answer, please 

## Load the program and display the program
The program is a 3-module system, each of which handles the urgency, sentiment and categories classification respectively

In [9]:
program = lb_math_metas[0].program[0]
program

predict = Predict(StringSignature(question -> reasoning, answer
    instructions='Solve the question and provide the answer in the correct format.'
    question = Field(annotation=str required=True json_schema_extra={'__dspy_field_type': 'input', 'prefix': 'Question:', 'desc': '${question}'})
    reasoning = Field(annotation=str required=True json_schema_extra={'prefix': "Reasoning: Let's think step by step in order to", 'desc': '${reasoning}', '__dspy_field_type': 'output'})
    answer = Field(annotation=str required=True json_schema_extra={'__dspy_field_type': 'output', 'prefix': 'Answer:', 'desc': '${answer}'})
))

## Define an evaluator and evaluate the base program

In [ ]:
import dspy
evaluate = dspy.Evaluate(
    devset=bench.test_set,
    metric=lb_math_metas[0].metric,
    num_threads=80,
    display_table=True,
    display_progress=True,
    max_errors=100 * len(bench.test_set)
)

In [ ]:
evaluate(program)

## Load the GEPA Optimizer

In [10]:
# Import GEPA and define the optimizer
from gepa_artifact.gepa.gepa import GEPA
from gepa_artifact.utils.capture_stream_logger import Logger

import time

runs_dir = os.path.join(os.getcwd(), "runs", time.strftime("%Y-%m-%d_%H-%M-%S"))
os.makedirs(runs_dir, exist_ok=True)

gepa_logger = Logger(os.path.join(runs_dir, "run_log.txt"))

if lb_math_metas[0].feedback_fn_maps is None or lb_math_metas[0].feedback_fn_maps[0] is None:
    def feedback_func(predictor_output, predictor_inputs, module_inputs, module_outputs, captured_trace):
        pred = lb_math_metas[0].metric_with_feedback(module_inputs, module_outputs, None)
        return {
            "feedback_score": pred.score,
            "feedback_text": pred.feedback,
        }

    feedback_fn_map = {k:feedback_func for k, v in program.named_predictors()}
else:
    feedback_fn_map = lb_math_metas[0].feedback_fn_maps[0]

optimizer = GEPA(
    named_predictor_to_feedback_fn_map=feedback_fn_map,
    knowledgebase_qe=None,
    metric=lb_math_metas[0].metric,
    run_linearized_gepa=False,
    use_merge=True, 
    teacher_lm = tlm,
    set_for_merge_minibatch='val', 
    track_scores_on='val',
    max_metric_calls=700,
    run_dir=runs_dir,
    logger=gepa_logger,
    num_threads=40
)

In [11]:
x = lb_math_metas[0].program[0].get_lm()
 
print(x)

None


## Optimize the program with GEPA

In [ ]:
optimized_program = optimizer.compile(
    lb_math_metas[0].program[0],
    trainset=bench.train_set,
    valset=bench.val_set[:len(bench.val_set)//2],
)

## Now, let's evaluate the optimized program

In [ ]:
evaluate(optimized_program)

GEPA was able to optimize the base program **from 57% score to 61% score** in just 9 iterations. With higher budget, the optimized program's score can go as high as **64%**.

### Let's print the prompts that GEPA discovered

In [ ]:
for name, pred in optimized_program.named_predictors():
    print("================================")
    print(f"Predictor: {name}")
    print("================================")
    print("Prompt:")
    print(pred.signature.instructions)
    print("*********************************")